# 01 - Environment Setup

## Learning objectives
1. Install the spatial-transcriptomics Python stack reproducibly.
2. Verify versions of the key libraries.
3. Understand the **AnnData** container by analogy to a medical-image study object.
4. Set random seeds for reproducibility.

## Concept
The ecosystem is built around three packages: **`anndata`** (the data container),
**`scanpy`** (single-cell/expression analysis), and **`squidpy`** (spatial extensions +
image handling). Around them we use the usual scientific-Python and imaging tools
(`numpy`, `pandas`, `scikit-image`, `opencv`, `scikit-learn`).


## Installing
Pick **one** approach. Conda is most reliable for the scientific stack; Python 3.10-3.11
is recommended.

**Conda (recommended):**
```bash
conda env create -f environment.yml
conda activate spatial-tx
python -m ipykernel install --user --name spatial-tx
```

**pip / venv:**
```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

If you prefer to install straight from the notebook, run the cell below (it is commented
out so it does not run by accident). Restart the kernel afterwards.


In [ ]:
# Optional in-notebook install. Uncomment to use, then restart the kernel.
# %pip install -r requirements.txt


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


## Version checks
Run this to confirm every required library imports and to record versions (useful when
asking for help or filing issues).

In [ ]:
import importlib

packages = [
    'scanpy', 'squidpy', 'anndata', 'numpy', 'pandas', 'scipy',
    'matplotlib', 'seaborn', 'sklearn', 'skimage', 'cv2',
]

print(f'{"package":<14} version')
print('-' * 28)
for name in packages:
    try:
        mod = importlib.import_module(name)
        print(f'{name:<14} {getattr(mod, "__version__", "(no __version__)")}')
    except Exception as exc:  # missing or broken install
        print(f'{name:<14} NOT IMPORTED -> {exc}')


**Expected output:** a table listing each package with a version string. If any row says
`NOT IMPORTED`, fix the install (see the README troubleshooting section) before continuing.


In [ ]:
import scanpy as sc

# Make scanpy's logging/plots tutorial-friendly.
sc.settings.verbosity = 1            # 0=errors only ... 3=hints
sc.settings.set_figure_params(dpi=80, facecolor='white')
sc.logging.print_header()


## Reproducibility
Stochastic steps (PCA initialization, UMAP, Leiden, train/test splits) can shift slightly
between runs. We seed everything via `st.set_seeds()` and additionally pass
`random_state=st.SEED` into scanpy calls later. This mirrors fixing seeds before training
a CNN so results are comparable.

In [ ]:
seed = st.set_seeds()  # already called in SETUP; safe to repeat
print('Global seed:', seed)


## Meet AnnData (by analogy)

An **AnnData** object is the ST equivalent of a well-organized imaging study object. For a
matrix of `n_obs` spots x `n_vars` genes:

| AnnData slot | Holds | Imaging analogy |
|---|---|---|
| `.X` | the count matrix (spots x genes) | the image intensities |
| `.obs` | per-spot metadata (a DataFrame) | per-voxel labels/measurements |
| `.var` | per-gene metadata (a DataFrame) | per-channel/feature descriptors |
| `.obsm` | per-spot arrays (e.g. `spatial`, `X_pca`, `X_umap`) | coordinate maps / embeddings |
| `.uns` | unstructured extras (the H&E image, scale factors) | study header / attached image |
| `.layers` | alternative matrices (e.g. raw counts vs normalized) | pre/post-processing volumes |

Let's build a tiny synthetic AnnData to see the structure with zero download.


In [ ]:
import numpy as np
import pandas as pd
import anndata as ad

rng = np.random.default_rng(st.SEED)
n_spots, n_genes = 6, 4
counts = rng.poisson(2.0, size=(n_spots, n_genes)).astype(float)

demo = ad.AnnData(
    X=counts,
    obs=pd.DataFrame({'total_counts': counts.sum(1)},
                     index=[f'spot_{i}' for i in range(n_spots)]),
    var=pd.DataFrame(index=[f'Gene{j}' for j in range(n_genes)]),
)
demo.obsm['spatial'] = rng.integers(0, 1000, size=(n_spots, 2))
print(demo)
print('\nX (spots x genes):\n', demo.X)
print('\n.obs:\n', demo.obs)
print('\n.obsm["spatial"]:\n', demo.obsm['spatial'])


**Expected output:** an `AnnData object with n_obs x n_vars = 6 x 4` summary, the 6x4
count matrix, an `.obs` table with `total_counts`, and a 6x2 array of fake coordinates.

## Common pitfalls
- Installing into the wrong environment, then running Jupyter from another - confirm the
  kernel matches `spatial-tx`.
- Forgetting to restart the kernel after an in-notebook `%pip install`.

## Interpretation
You now have a working toolkit and a mental model of AnnData: one object that bundles the
expression matrix, spot/gene metadata, coordinates, and the histology image.

## What this means biologically
Nothing biological yet - but a clean, reproducible environment is what separates a
research-grade analysis from an irreproducible one. Seeds and pinned versions are part of
good experimental hygiene.

---
**Next:** `02_fetch_public_visium_data.ipynb` - download a real Visium dataset.
